# NB03: Model Internals

Analyze v0 and v1 character-level models from checkpoints.

Analyses:
1. Character embedding PCA/t-SNE/UMAP (do vowels cluster? punctuation?)
2. Attention heatmaps per layer/head
3. Layer-wise activation analysis (CKA similarity)
4. v0 vs v1 comparison: what did ALiBi + longer context learn differently?

No API dependencies. Runs entirely from local checkpoints.

Depends on: NB00 (fingerprint for context), local checkpoints

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from lib.config import CHECKPOINTS_DIR, RESULTS_DIR

sns.set_theme(style='whitegrid', font_scale=1.1)

INTERNALS_DIR = RESULTS_DIR / 'internals'
INTERNALS_DIR.mkdir(parents=True, exist_ok=True)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Checkpoints: {list(CHECKPOINTS_DIR.glob("*.pt"))}')

## 1. Load Models

In [ ]:
import pickle

# Load meta for character mappings
meta_path = Path('..') / 'data' / 'meta.pkl'
with open(meta_path, 'rb') as f:
    meta = pickle.load(f)
itos = meta['itos']
stoi = meta['stoi']
vocab_size = len(itos)
print(f'Vocab size: {vocab_size}')
print(f'Characters: {[itos[i] for i in range(min(20, vocab_size))]}')

# Load v0 model
from models.v0.model import McCarthyGPT as V0Model

v0_path = CHECKPOINTS_DIR / 'best.pt'
v0_ckpt = torch.load(v0_path, map_location=device, weights_only=False)
v0_config = v0_ckpt['config']
v0_model = V0Model(v0_config).to(device)
v0_model.load_state_dict(v0_ckpt['model'])
v0_model.eval()
print(f'\nv0: {sum(p.numel() for p in v0_model.parameters()):,} params')
print(f'v0 config: {v0_config}')

# Load v1 model
from models.v1.model import McCarthyGPT as V1Model

v1_path = CHECKPOINTS_DIR / 'best_modal.pt'
v1_ckpt = torch.load(v1_path, map_location=device, weights_only=False)
v1_config = v1_ckpt['config']
v1_model = V1Model(v1_config).to(device)
v1_model.load_state_dict(v1_ckpt['model'])
v1_model.eval()
print(f'\nv1: {sum(p.numel() for p in v1_model.parameters()):,} params')
print(f'v1 config: {v1_config}')

## 2. Character Embedding Analysis

In [ ]:
# Extract character embeddings from both models
v0_emb = v0_model.tok_emb.weight.detach().cpu().numpy()
v1_emb = v1_model.tok_emb.weight.detach().cpu().numpy()

print(f'v0 embeddings: {v0_emb.shape}')
print(f'v1 embeddings: {v1_emb.shape}')

# Character categories for coloring
vowels = set('aeiouAEIOU')
consonants = set('bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ')
punctuation = set('.,:;\'"!?-()[]')
digits = set('0123456789')
whitespace = set(' \t\n\r')

def char_category(ch):
    if ch in vowels: return 'vowel'
    if ch in consonants: return 'consonant'
    if ch in punctuation: return 'punctuation'
    if ch in digits: return 'digit'
    if ch in whitespace: return 'whitespace'
    return 'other'

categories = [char_category(itos[i]) for i in range(vocab_size)]
cat_colors = {
    'vowel': '#e74c3c',
    'consonant': '#3498db',
    'punctuation': '#2ecc71',
    'digit': '#f39c12',
    'whitespace': '#95a5a6',
    'other': '#9b59b6',
}
colors = [cat_colors[c] for c in categories]

In [ ]:
# PCA + t-SNE for both models
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for col, (emb, name) in enumerate([(v0_emb, 'v0'), (v1_emb, 'v1')]):
    # PCA
    pca = PCA(n_components=2)
    emb_pca = pca.fit_transform(emb)

    ax = axes[0, col]
    for cat in cat_colors:
        mask = [c == cat for c in categories]
        ax.scatter(emb_pca[mask, 0], emb_pca[mask, 1],
                   c=cat_colors[cat], label=cat, alpha=0.7, s=60)
    # Label characters
    for i in range(vocab_size):
        ch = itos[i]
        if ch.strip():  # skip whitespace labels
            ax.annotate(repr(ch)[1:-1], (emb_pca[i, 0], emb_pca[i, 1]),
                        fontsize=7, alpha=0.8)
    ax.set_title(f'{name} Character Embeddings (PCA)')
    ax.legend(fontsize=8, loc='lower right')

    # t-SNE
    tsne = TSNE(n_components=2, perplexity=min(15, vocab_size - 1),
                random_state=42, n_iter=1000)
    emb_tsne = tsne.fit_transform(emb)

    ax = axes[1, col]
    for cat in cat_colors:
        mask = [c == cat for c in categories]
        ax.scatter(emb_tsne[mask, 0], emb_tsne[mask, 1],
                   c=cat_colors[cat], label=cat, alpha=0.7, s=60)
    for i in range(vocab_size):
        ch = itos[i]
        if ch.strip():
            ax.annotate(repr(ch)[1:-1], (emb_tsne[i, 0], emb_tsne[i, 1]),
                        fontsize=7, alpha=0.8)
    ax.set_title(f'{name} Character Embeddings (t-SNE)')
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Character Embedding Geometry: v0 vs v1', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(INTERNALS_DIR / 'char_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. UMAP Embedding Visualization

In [ ]:
from umap import UMAP

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for col, (emb, name) in enumerate([(v0_emb, 'v0'), (v1_emb, 'v1')]):
    reducer = UMAP(n_components=2, n_neighbors=min(10, vocab_size - 1),
                   min_dist=0.3, random_state=42)
    emb_umap = reducer.fit_transform(emb)

    ax = axes[col]
    for cat in cat_colors:
        mask = [c == cat for c in categories]
        ax.scatter(emb_umap[mask, 0], emb_umap[mask, 1],
                   c=cat_colors[cat], label=cat, alpha=0.7, s=60)
    for i in range(vocab_size):
        ch = itos[i]
        if ch.strip():
            ax.annotate(repr(ch)[1:-1], (emb_umap[i, 0], emb_umap[i, 1]),
                        fontsize=7, alpha=0.8)
    ax.set_title(f'{name} Character Embeddings (UMAP)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(INTERNALS_DIR / 'char_embeddings_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Attention Heatmaps

In [ ]:
# Generate attention patterns for a sample input
sample_text = 'They rode on and the sun in the east'
sample_ids = torch.tensor([[stoi[c] for c in sample_text]], device=device)
seq_len = sample_ids.shape[1]

print(f'Input: "{sample_text}"')
print(f'Token IDs: {sample_ids[0].tolist()}')
print(f'Sequence length: {seq_len}')


def get_attention_weights(model, x):
    """Extract attention weights from all layers/heads.
    Returns list of (n_heads, seq_len, seq_len) tensors, one per layer."""
    attentions = []

    # Hook into attention layers
    hooks = []
    def make_hook(layer_idx):
        def hook_fn(module, input, output):
            # output is attention weights if we modify forward
            # For now, we'll compute attention manually
            pass
        return hook_fn

    # Manual forward to capture attention
    with torch.no_grad():
        # Get embeddings
        tok_emb = model.tok_emb(x)
        if hasattr(model, 'pos_emb'):
            pos_emb = model.pos_emb(torch.arange(x.shape[1], device=x.device))
            h = model.drop(tok_emb + pos_emb)
        else:
            h = model.drop(tok_emb)

        for block in model.blocks:
            # Self-attention
            attn = block.attn
            B, T, C = h.shape

            # Get Q, K, V
            if hasattr(attn, 'qkv'):
                qkv = attn.qkv(block.ln1(h))
                n_head = attn.n_head
                head_dim = C // n_head
                q, k, v = qkv.split(C, dim=-1)
            elif hasattr(attn, 'c_attn'):
                qkv = attn.c_attn(block.ln1(h))
                n_head = attn.n_head
                head_dim = C // n_head
                q, k, v = qkv.split(C, dim=-1)
            else:
                print('Unknown attention architecture, skipping')
                break

            q = q.view(B, T, n_head, head_dim).transpose(1, 2)
            k = k.view(B, T, n_head, head_dim).transpose(1, 2)

            # Attention weights
            scale = head_dim ** -0.5
            weights = (q @ k.transpose(-2, -1)) * scale

            # Causal mask
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            weights.masked_fill_(mask, float('-inf'))
            weights = torch.softmax(weights, dim=-1)

            attentions.append(weights[0].cpu().numpy())  # (n_head, T, T)

            # Continue forward pass
            h = h + block.attn(block.ln1(h))
            h = h + block.ff(block.ln2(h))

    return attentions


print('Extracting attention weights...')
v0_attn = get_attention_weights(v0_model, sample_ids)
v1_attn = get_attention_weights(v1_model, sample_ids)
print(f'v0: {len(v0_attn)} layers, v1: {len(v1_attn)} layers')

In [ ]:
# Plot attention heatmaps for v0
char_labels = list(sample_text)
n_layers = len(v0_attn)
n_heads = v0_attn[0].shape[0]

fig, axes = plt.subplots(n_layers, n_heads, figsize=(4 * n_heads, 4 * n_layers))
if n_layers == 1:
    axes = axes[np.newaxis, :]
if n_heads == 1:
    axes = axes[:, np.newaxis]

for layer in range(n_layers):
    for head in range(n_heads):
        ax = axes[layer, head]
        sns.heatmap(v0_attn[layer][head], ax=ax, cmap='viridis',
                    xticklabels=char_labels, yticklabels=char_labels,
                    cbar=False, square=True)
        ax.set_title(f'L{layer} H{head}', fontsize=9)
        ax.tick_params(labelsize=7)

plt.suptitle('v0 Attention Patterns', fontsize=14)
plt.tight_layout()
plt.savefig(INTERNALS_DIR / 'v0_attention.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot attention heatmaps for v1
n_layers_v1 = len(v1_attn)
n_heads_v1 = v1_attn[0].shape[0]

fig, axes = plt.subplots(n_layers_v1, n_heads_v1,
                         figsize=(3 * n_heads_v1, 3 * n_layers_v1))
if n_layers_v1 == 1:
    axes = axes[np.newaxis, :]
if n_heads_v1 == 1:
    axes = axes[:, np.newaxis]

for layer in range(n_layers_v1):
    for head in range(n_heads_v1):
        ax = axes[layer, head]
        sns.heatmap(v1_attn[layer][head], ax=ax, cmap='viridis',
                    xticklabels=char_labels, yticklabels=char_labels,
                    cbar=False, square=True)
        ax.set_title(f'L{layer} H{head}', fontsize=8)
        ax.tick_params(labelsize=6)

plt.suptitle('v1 Attention Patterns (ALiBi)', fontsize=14)
plt.tight_layout()
plt.savefig(INTERNALS_DIR / 'v1_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Layer-wise CKA Similarity (v0 vs v1)

In [ ]:
def get_layer_activations(model, x):
    """Get activations after each transformer block."""
    activations = []
    with torch.no_grad():
        tok_emb = model.tok_emb(x)
        if hasattr(model, 'pos_emb'):
            pos_emb = model.pos_emb(torch.arange(x.shape[1], device=x.device))
            h = model.drop(tok_emb + pos_emb)
        else:
            h = model.drop(tok_emb)

        activations.append(h.cpu().numpy().reshape(-1, h.shape[-1]))
        for block in model.blocks:
            h = h + block.attn(block.ln1(h))
            h = h + block.ff(block.ln2(h))
            activations.append(h.cpu().numpy().reshape(-1, h.shape[-1]))
    return activations


def linear_cka(X, Y):
    """Linear CKA similarity between two activation matrices."""
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    hsic_xy = np.linalg.norm(X.T @ Y, 'fro') ** 2
    hsic_xx = np.linalg.norm(X.T @ X, 'fro') ** 2
    hsic_yy = np.linalg.norm(Y.T @ Y, 'fro') ** 2
    if hsic_xx * hsic_yy == 0:
        return 0.0
    return hsic_xy / np.sqrt(hsic_xx * hsic_yy)


# Use a longer sample for CKA
cka_text = 'They rode on and the sun in the east flushed pale streaks of light'
cka_ids = torch.tensor([[stoi[c] for c in cka_text]], device=device)

v0_acts = get_layer_activations(v0_model, cka_ids)
v1_acts = get_layer_activations(v1_model, cka_ids)

print(f'v0 layers: {len(v0_acts)}, v1 layers: {len(v1_acts)}')

# CKA matrix: v0 layers x v1 layers
cka_matrix = np.zeros((len(v0_acts), len(v1_acts)))
for i in range(len(v0_acts)):
    for j in range(len(v1_acts)):
        cka_matrix[i, j] = linear_cka(v0_acts[i], v1_acts[j])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cka_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f'v1 L{i}' for i in range(len(v1_acts))],
            yticklabels=[f'v0 L{i}' for i in range(len(v0_acts))],
            ax=ax)
ax.set_title('CKA Similarity: v0 vs v1 Layer Representations')
ax.set_xlabel('v1 (ALiBi + SwiGLU, 8L/8H/512D)')
ax.set_ylabel('v0 (Standard, 6L/6H/384D)')
plt.tight_layout()
plt.savefig(INTERNALS_DIR / 'cka_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Embedding Similarity Analysis

Which characters does each model consider most similar?

In [ ]:
from scipy.spatial.distance import cosine

def top_similar(emb, idx, itos, k=5):
    """Find k most similar characters to idx by cosine similarity."""
    sims = []
    for j in range(len(itos)):
        if j == idx:
            continue
        sim = 1 - cosine(emb[idx], emb[j])
        sims.append((j, itos[j], sim))
    sims.sort(key=lambda x: -x[2])
    return sims[:k]


# Compare what each model thinks is similar to key characters
probe_chars = ['a', 'e', 't', '.', ',', ' ', 'A', 'T']

for ch in probe_chars:
    if ch not in stoi:
        continue
    idx = stoi[ch]
    print(f'\nMost similar to {repr(ch)}:')

    v0_sim = top_similar(v0_emb, idx, itos)
    v1_sim = top_similar(v1_emb, idx, itos)

    print(f'  v0: {[(repr(c), f"{s:.3f}") for _, c, s in v0_sim]}')
    print(f'  v1: {[(repr(c), f"{s:.3f}") for _, c, s in v1_sim]}')

## 7. Summary

In [ ]:
print('=== MODEL INTERNALS SUMMARY ===')
print(f'\nv0: {sum(p.numel() for p in v0_model.parameters()):,} params')
print(f'  Architecture: {v0_config}')
print(f'  Embedding dim: {v0_emb.shape[1]}')
print(f'  Layers: {len(v0_attn)}, Heads: {v0_attn[0].shape[0]}')

print(f'\nv1: {sum(p.numel() for p in v1_model.parameters()):,} params')
print(f'  Architecture: {v1_config}')
print(f'  Embedding dim: {v1_emb.shape[1]}')
print(f'  Layers: {len(v1_attn)}, Heads: {v1_attn[0].shape[0]}')

print(f'\nOutputs saved to: {INTERNALS_DIR}')
print(f'  - char_embeddings.png')
print(f'  - char_embeddings_umap.png')
print(f'  - v0_attention.png')
print(f'  - v1_attention.png')
print(f'  - cka_similarity.png')